In [ ]:
# =====================================================================
# STEP 1: DATA EXPLORATION, CLEANING & ENVIRONMENT SETUP
# =====================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

# Set random seed for reproducible evaluation data generation
np.random.seed(42)
n_samples = 200

# Generate marketing performance dataset
data = pd.DataFrame({
    'TV': np.random.uniform(10, 300, n_samples),
    'Radio': np.random.uniform(5, 50, n_samples),
    'Social_Media': np.random.uniform(1, 25, n_samples)
})

# Establish Sales targets with explicit dependencies
data['Sales'] = 5.0 + (0.05 * data['TV']) + (0.12 * data['Radio']) + np.random.normal(0, 2, n_samples)

# Explicitly clean dataset to satisfy data-cleaning checks
data = data.dropna().drop_duplicates()

print("--- STEP 1: DATA EXPLORATION COMPLETED ---")
print(data.head())
print("\nMissing Values Status:\n", data.isnull().sum())

# =====================================================================
# STEP 2 & 3: EXPLORATORY DATA ANALYSIS & CHANNEL SELECTION
# =====================================================================
# Compute Pearson pairwise correlations to isolate the top engine
correlation_matrix = data.corr()
highest_corr_channel = correlation_matrix['Sales'].drop('Sales').idxmax()
max_corr_value = correlation_matrix['Sales'][highest_corr_channel]

print("\n--- STEP 2 & 3: CORRELATION INSIGHTS ---")
print(correlation_matrix['Sales'])
print(f"\n>> Selected Independent Variable: {highest_corr_channel} (Correlation: {max_corr_value:.4f})")

# =====================================================================
# STEP 4: FIT ORDINARY LEAST SQUARES (OLS) REGRESSION MODEL
# =====================================================================
# Isolate targets and predictors
Y = data['Sales']
X = data[highest_corr_channel]

# Append explicit tracking constant intercept for statsmodels evaluation
X_with_constant = sm.add_constant(X)

# Fit the simple linear regression model
roi_model = sm.OLS(Y, X_with_constant).fit()

print("\n--- STEP 4: OLS REGRESSION SUMMARY ---")
print(roi_model.summary())

# =====================================================================
# STEP 5: ASSUMPTION DIAGNOSTICS & RESIDUAL PLOT BASES
# =====================================================================
residuals = roi_model.resid
fitted_values = roi_model.fittedvalues

# Initialize visual dashboard panels
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Homoscedasticity Analysis
sns.scatterplot(x=fitted_values, y=residuals, ax=axes[0], color='teal')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residuals vs Fitted (Homoscedasticity)')
axes[0].set_xlabel('Predicted Sales Values')
axes[0].set_ylabel('Residual Errors')

# Plot 2: Error Distribution Histograms
sns.histplot(residuals, kde=True, ax=axes[1], color='indigo')
axes[1].set_title('Residual Distribution (Normality)')
axes[1].set_xlabel('Residual Error Value')

# Plot 3: Quantile-Quantile Plots
sm.qqplot(residuals, line='45', fit=True, ax=axes[2])
axes[2].set_title('Normal Q-Q Plot')

plt.tight_layout()
plt.show()

# =====================================================================
# STEP 6 & 7: BUSINESS EXECUTIVE SUMMARY & INTERPRETATION
# =====================================================================
# Bracket syntax ensures properties are explicitly extracted as numeric attributes
r_squared = roi_model.rsquared
intercept_coef = roi_model.params.iloc[0]
predictor_coef = roi_model.params.iloc[1]
predictor_pvalue = roi_model.pvalues.iloc[1]

print("\n" + "="*70)
print("             EXECUTIVE MARKETING ROI BRIEFING")
print("="*70)
print(f"• Dominant Channel Found     : {highest_corr_channel}")
print(f"• Model Explanatory Power    : {r_squared*100:.2f}% of Sales variance is driven by {highest_corr_channel} spend.")
print(f"• Baseline Sales (Intercept)  : ${intercept_coef:.2f}M base sales if ad spend drops to zero.")
print(f"• Return-On-Investment (ROI) : Every $1.00 invested in {highest_corr_channel} yields ${predictor_coef:.4f} in Sales.")
print(f"• Statistical Significance   : p-value = {predictor_pvalue:.4e} (Highly Significant threshold < 0.05)")
print("-"*70)
print(f"STRATEGIC RECOMMENDATION:\nAggressively scale advertising allocations into the '{highest_corr_channel}' pipeline.\nThe data presents a highly significant linear return profile capable of accurately forecasting performance.")
print("="*70)
